# Feature store Travel Agency recommendation
Puede crear FS de dos formas
- Adicione a una tabla PK. 
- Usar la función `FeatureEngineeringClient`

Vamos a crear dos tablas de caracteristicas:
* **User features**: features for a given user in a given point in time (location, previous purchases if any, tenure days etc)
* **Destination features**: data on the travel destination for a given point in time (interest tracked by the number of clicks & impression)

### Soporte para punto en el tiempo para tablas de caracteristicas
Databricks FS soporta casos de usos que requiere exactitud en un punto en el tiempo. Los datos a menudo tienen dependencia del tiempo. Para este ejemplo adicionamos rolling-window features. Al construir el modelo, debemos considerar solo valores de características hasta el tiempo de los valores observados. De lo contrario puede tomar valores después de la marca de tiempo de los valores a entrenar. Esto es llamado fuga de datos “data leakage” y puede afectar negativamente el desempeño del modelo. Tablas de caracteristicas serie de tiempo incluye una columna clave (key) para identificar el tiempo.

### Calcular las caracteristicas
La caracteristica usuario captura el perfil de usuario como precio de compras anteriores. Porque los datos de reserva no cambian muy amenudo, el puede ser computado una ves por día en batch.
La característica destino incluye popularidad como las impresiones y clicks, tambien precio y tiempo de reserva

In [0]:
catalog = "main"
schema = "dbdemos_fs_travel"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"""USE `{catalog}`.`{schema}`""")

In [0]:
from pyspark.sql import functions as F, Window as W

def create_user_behavior_features(travel_purchase_df):
    """
    Compute user-level rolling window behavior features.
    """
    travel_purchase_df = travel_purchase_df.withColumn("ts_l", F.col("ts").cast("long"))

    user_behavior_df = (
        travel_purchase_df
        .withColumn(
            "lookedup_price_7d_rolling_sum",F.sum("price").over(W.partitionBy("user_id").orderBy(F.col("ts_l")).rangeBetween(-7 * 86400, 0)))
        .withColumn(
            "lookups_7d_rolling_sum",F.count("*").over(W.partitionBy("user_id").orderBy(F.col("ts_l")).rangeBetween(-7 * 86400, 0)))
        .withColumn(
            "mean_price_7d",F.when(F.col("lookups_7d_rolling_sum") > 0,
                   F.col("lookedup_price_7d_rolling_sum") / F.col("lookups_7d_rolling_sum")).otherwise(F.lit(None)))
        .withColumn("tickets_purchased", F.col("purchased").cast("int"))
        .withColumn(
            "last_6m_purchases",F.sum("tickets_purchased").over(W.partitionBy("user_id").orderBy(F.col("ts_l")).rangeBetween(-6 * 30 * 86400, 0)).cast("double"))
        .select("user_id", "ts", "mean_price_7d", "last_6m_purchases")
    )
    return user_behavior_df

In [0]:
def create_user_features(travel_purchase_df, user_demography_df):
    """
    Combine user-level rolling behavior features with demographic info.
    Adds tenure_days = datediff(ts, first_login_date).
    """
    # Compute behavioral features
    user_behavior_df = create_user_behavior_features(travel_purchase_df)

    # Join static demographics
    combined_df = (
        user_behavior_df
        .join(user_demography_df, on="user_id", how="left")
        # tenure_days = event time - first login date
        .withColumn(
            "tenure_days",
            F.datediff(F.to_date("ts"), F.col("first_login_date")).cast(LongType())
        )
        .select(
            "user_id",
            "ts",
            "mean_price_7d",
            "last_6m_purchases",
            "age",
            "gender",
            "income_bracket",
            "loyalty_tier",
            "billing_state",
            "billing_city",
            "tenure_days"
        )
    )

    return combined_df

# Example use
user_demography_df = spark.table(f"{catalog}.{schema}.user_demography")
travel_purchase_df = spark.table("travel_purchase")

user_features_df = create_user_features(travel_purchase_df, user_demography_df)
display(user_features_df.limit(5))

user_id,ts,mean_price_7d,last_6m_purchases,age,gender,income_bracket,loyalty_tier,billing_state,billing_city,tenure_days
2,2022-08-02T00:17:10.875Z,1998.0,0.0,39,F,high,silver,TX,Dallas,-983
3,2022-08-06T18:23:58.190Z,2586.97,0.0,64,M,high,silver,NY,Albany,-924
5,2022-08-05T19:24:39.411Z,1584.0,0.0,34,F,medium,platinum,RI,Providence,-477
11,2022-08-03T08:20:07.445Z,1584.0,0.0,53,F,low,platinum,GA,Augusta,-573
12,2022-08-07T23:12:14.713Z,2214.0,0.0,38,Other,high,platinum,CO,Colorado Springs,-867
14,2022-08-03T11:31:29.184Z,6268.03,0.0,48,F,low,silver,SD,Sioux Falls,-523
15,2022-08-04T11:15:21.817Z,7116.39,0.0,40,F,high,platinum,FL,Jacksonville,-328
16,2022-08-02T09:55:56.655Z,1998.0,0.0,23,Other,medium,gold,MI,Ann Arbor,-30
16,2022-08-04T11:05:53.669Z,3090.705,1.0,23,Other,medium,gold,MI,Ann Arbor,-28
20,2022-08-03T12:26:06.226Z,4104.0,0.0,57,M,medium,gold,MS,Jackson,-320


In [0]:
def create_destination_features(travel_purchase_df):
    """
    Computes the destination_features feature group.
    """
    return (
        travel_purchase_df
          .withColumn("clicked", F.col("clicked").cast("int"))
          .withColumn("sum_clicks_7d", 
            F.sum("clicked").over(w.Window.partitionBy("destination_id").orderBy(F.col("ts").cast("long")).rangeBetween(start=-(7 * 86400), end=0))
          )
          .withColumn("sum_impressions_7d", 
            F.count("*").over(w.Window.partitionBy("destination_id").orderBy(F.col("ts").cast("long")).rangeBetween(start=-(7 * 86400), end=0))
          )
          .select("destination_id", "ts", "sum_clicks_7d", "sum_impressions_7d")
    )  
destination_features_df = create_destination_features(spark.table('travel_purchase'))
display(destination_features_df.limit(5))

destination_id,ts,sum_clicks_7d,sum_impressions_7d
0,2022-07-31T18:40:47.379Z,1,1
0,2022-07-31T21:11:16.870Z,2,2
0,2022-08-01T04:32:23.711Z,3,3
0,2022-08-01T05:52:27.751Z,4,4
0,2022-08-01T07:22:32.130Z,5,5
0,2022-08-01T08:21:09.723Z,5,6
0,2022-08-01T08:33:25.110Z,5,7
0,2022-08-01T09:15:15.544Z,6,8
0,2022-08-01T09:55:29.553Z,7,9
0,2022-08-01T10:01:34.777Z,8,10


### Crear Feature Table
Usamos FeatureStore client para creear tablas FS. Note el parametro `timestamp_keys='ts'` usado como columna de tiempo. Databricks Feature Store usa esta informacion para filtrar features y prevenir potenciales fugas de datos (leakage).

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient(model_registry_uri="databricks-uc")
# help(fe.create_table)

# first create a table with User Features calculated above 
fe_table_name_users = f"{catalog}.{schema}.user_features_advanced"
#fe.drop_table(name=fe_table_name_users)
fe.create_table(
    name=fe_table_name_users, # unique table name (in case you re-run the notebook multiple times)
    primary_keys=["user_id", "ts"],
    timestamp_keys="ts",
    df=user_features_df,
    description="User-level rolling behavior + demographic features with tenure tracking"
    # attach tags if necessary
    #,tags={"demo":"yes"}
)

2026/04/20 17:01:20 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['user_id', 'ts'] of table 'main.dbdemos_fs_travel.user_features_advanced' to NOT NULL.
2026/04/20 17:01:23 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['user_id', 'ts'] on table 'main.dbdemos_fs_travel.user_features_advanced'.
2026/04/20 17:01:36 INFO databricks.ml_features._spark_client._spark_client: Feature Engineering client applies Z-ordering to this table by default. You may want to explore Liquid Clustering for better performance.
2026/04/20 17:01:39 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'main.dbdemos_fs_travel.user_features_advanced'.


<FeatureTable: name='main.dbdemos_fs_travel.user_features_advanced', table_id='80b4d8ea-2ad5-49df-bd2e-c18767827b85', description='User-level rolling behavior + demographic features with tenure tracking', primary_keys=['user_id', 'ts'], partition_columns=[], features=['user_id',
 'ts',
 'mean_price_7d',
 'last_6m_purchases',
 'age',
 'gender',
 'income_bracket',
 'loyalty_tier',
 'billing_state',
 'billing_city',
 'tenure_days'], creation_timestamp=1776704479037, online_stores=[], notebook_producers=[], job_producers=[], table_data_sources=[], path_data_sources=[], custom_data_sources=[], timestamp_keys=['ts'], tags={}>

In [0]:
fe_table_name_destinations = f"{catalog}.{schema}.destination_features_advanced"
# second create another Feature Table from popular Destinations
# for the second table, we show how to create and write as two separate operations
# fe.drop_table(name=fe_table_name_destinations)
fe.create_table(
    name=fe_table_name_destinations, # unique table name (in case you re-run the notebook multiple times)
    primary_keys=["destination_id", "ts"],
    timestamp_keys="ts", 
    schema=destination_features_df.schema,
    description="Destination Popularity Features"
    # attach tags if necessary
    #,tags={"demo":"yes"}
)
fe.write_table(name=fe_table_name_destinations, df=destination_features_df)

2026/04/20 17:16:22 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['destination_id', 'ts'] of table 'main.dbdemos_fs_travel.destination_features_advanced' to NOT NULL.
2026/04/20 17:16:25 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['destination_id', 'ts'] on table 'main.dbdemos_fs_travel.destination_features_advanced'.
2026/04/20 17:16:27 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'main.dbdemos_fs_travel.destination_features_advanced'.
2026/04/20 17:16:43 INFO databricks.ml_features._spark_client._spark_client: Feature Engineering client applies Z-ordering to this table by default. You may want to explore Liquid Clustering for better performance.


In [0]:
%sql
-- User feature table
ALTER TABLE user_features_advanced SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE user_features_advanced ALTER COLUMN user_id SET NOT NULL;

-- Destination feature table
ALTER TABLE destination_features_advanced SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');
ALTER TABLE destination_features_advanced ALTER COLUMN destination_id SET NOT NULL;

2026-04-20 19:41:23,889 7827 INFO execute_command Execute command for command <Truncated message due to truncation error>
2026-04-20 19:41:23,889 7827 INFO execute_command Execute command for command <Truncated message due to truncation error>
INFO:pyspark.sql.connect.client.logging:Execute command for command <Truncated message due to truncation error>
2026-04-20 19:41:23,908 7827 INFO _execute_and_fetch ExecuteAndFetch
2026-04-20 19:41:23,908 7827 INFO _execute_and_fetch ExecuteAndFetch
INFO:pyspark.sql.connect.client.logging:ExecuteAndFetch
2026-04-20 19:41:23,911 7827 INFO _execute_and_fetch_as_iterator ExecuteAndFetchAsIterator
2026-04-20 19:41:23,911 7827 INFO _execute_and_fetch_as_iterator ExecuteAndFetchAsIterator
INFO:pyspark.sql.connect.client.logging:ExecuteAndFetchAsIterator
2026-04-20 19:41:25,951 7827 INFO schema Schema for plan: root { common { plan_id: 317 } local_relation { data: "\377\377\377\377@\000\000\000[truncated]" } }
2026-04-20 19:41:25,951 7827 INFO schema Sc